In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import VotingClassifier
import joblib
import warnings

warnings.filterwarnings("ignore")


In [5]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df.head()


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [8]:
# Drop 'id' column
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Handle BMI missing/zero values
df['bmi'] = df['bmi'].replace(0, pd.NA)
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             5110 non-null   object 
 1   age                5110 non-null   float64
 2   hypertension       5110 non-null   int64  
 3   heart_disease      5110 non-null   int64  
 4   ever_married       5110 non-null   object 
 5   work_type          5110 non-null   object 
 6   Residence_type     5110 non-null   object 
 7   avg_glucose_level  5110 non-null   float64
 8   bmi                5110 non-null   float64
 9   smoking_status     5110 non-null   object 
 10  stroke             5110 non-null   int64  
dtypes: float64(3), int64(3), object(5)
memory usage: 439.3+ KB


In [10]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.600000,1,True,False,True,False,True,False,False,True,True,False,False
1,61.0,0,0,202.21,28.893237,1,False,False,True,False,False,True,False,False,False,True,False
2,80.0,0,1,105.92,32.500000,1,True,False,True,False,True,False,False,False,False,True,False
3,49.0,0,0,171.23,34.400000,1,False,False,True,False,True,False,False,True,False,False,True
4,79.0,1,0,174.12,24.000000,1,False,False,True,False,False,True,False,False,False,True,False


In [12]:
X = df_encoded.drop('stroke', axis=1)
y = df_encoded['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)


Training set size: (4088, 16)
Test set size: (1022, 16)


In [14]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit only on training data
X_test_scaled = scaler.transform(X_test)        # Transform test data
joblib.dump(scaler, 'scaler.pkl')               # Save scaler


['scaler.pkl']

In [16]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)


LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [17]:
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)


DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [20]:
rf = RandomForestClassifier(
    n_estimators=800,
    class_weight='balanced',
    max_depth=None,
    random_state=42
)
rf.fit(X_train_scaled, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=800,
                       random_state=42)

In [23]:
for name, model in [("Logistic Regression", lr), ("Decision Tree", dt), ("Random Forest", rf)]:
    pred = model.predict(X_test)
    print(f"=== {name} ===")
    print("Accuracy:", round(accuracy_score(y_test, pred),4))
    print(classification_report(y_test, pred, zero_division=0))
    print()


=== Logistic Regression ===
Accuracy: 0.7466
              precision    recall  f1-score   support

           0       0.99      0.74      0.85       972
           1       0.14      0.80      0.24        50

    accuracy                           0.75      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.75      0.82      1022


=== Decision Tree ===
Accuracy: 0.9227
              precision    recall  f1-score   support

           0       0.96      0.96      0.96       972
           1       0.15      0.12      0.13        50

    accuracy                           0.92      1022
   macro avg       0.55      0.54      0.55      1022
weighted avg       0.92      0.92      0.92      1022


=== Random Forest ===
Accuracy: 0.9511
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   

In [30]:
# ------------------ Ensemble Model ------------------
ensemble_model = VotingClassifier(
    estimators=[
        ('lr', lr),
        ('dt', dt),
        ('rf', rf)
    ],
    voting='soft'  # Uses predicted probabilities for better accuracy
)

# Train the ensemble model
ensemble_model.fit(X_train_scaled, y_train)

VotingClassifier(estimators=[('lr',
                              LogisticRegression(class_weight='balanced',
                                                 max_iter=1000,
                                                 random_state=42)),
                             ('dt',
                              DecisionTreeClassifier(class_weight='balanced',
                                                     random_state=42)),
                             ('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     n_estimators=800,
                                                     random_state=42))],
                 voting='soft')

In [35]:
# ------------------ Evaluate Ensemble Model ------------------
ensemble_pred = ensemble_model.predict(X_test_scaled)
print("=== Ensemble Model ===")
print("Accuracy:", round(accuracy_score(y_test, ensemble_pred), 4))
print(classification_report(y_test, ensemble_pred, zero_division=0))
print()

=== Ensemble Model ===
Accuracy: 0.9286
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       972
           1       0.17      0.12      0.14        50

    accuracy                           0.93      1022
   macro avg       0.56      0.55      0.55      1022
weighted avg       0.92      0.93      0.92      1022




In [27]:
joblib.dump(lr, "model_lr.pkl")
joblib.dump(dt, "model_dt.pkl")
joblib.dump(rf, "model_rf.pkl")
joblib.dump(list(X.columns), "model_columns.pkl")
joblib.dump(ensemble_model, "model_ensemble.pkl")

print("Saved models and feature columns!")


Saved models and feature columns!


In [38]:
import numpy as np

# Example input (High Risk)
sample = {
    "age": 65,
    "avg_glucose_level": 280,
    "bmi": 42,
    "hypertension": 1,
    "heart_disease": 1,
    "gender_Male": 1,
    "ever_married_Yes": 1,
    "work_type_Private": 1,
    "Residence_type_Urban": 1,
    "smoking_status_never smoked": 1
}

# Convert to DataFrame
sample_df = pd.DataFrame([sample])
sample_df = sample_df.reindex(columns=X.columns, fill_value=0)

# Scale input
sample_scaled = scaler.transform(sample_df)

# Predict using Random Forest
prob = rf.predict_proba(sample_scaled)[0][1]
print(f"Predicted Stroke Probability: {prob:.2f}")

if prob < 0.05:
    print("✅ Very Low Risk")
elif prob < 0.15:
    print("✅ Low Risk")
elif prob < 0.30:
    print("⚠️ Medium Risk")
elif prob < 0.50:
    print("⚠️ High Risk")
else:
    print("🚨 Very High Risk")

Predicted Stroke Probability: 0.18
⚠️ Medium Risk
